In [2]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)

Spark version: 4.0.1


# **Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng**

In [8]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupation.txt")

In [9]:
occupation_rdd.collect()

['1,Programmer',
 '2,Doctor',
 '3,Engineer',
 '4,Teacher',
 '5,Lawyer',
 '6,Artist',
 '7,Manager',
 '8,Nurse',
 '9,Salesperson',
 '10,Accountant',
 '11,Journalist',
 '12,Designer',
 '13,Researcher',
 '14,Consultant',
 '15,Student']

In [12]:
user_rdd.take(5)

['1,M,28,3,12345',
 '2,F,35,7,23456',
 '3,M,42,2,34567',
 '4,F,19,10,45678',
 '5,M,31,1,56789']

In [13]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

In [14]:
#Map user lay userId va occupationId
user_mapped = user_rdd.map(lambda x: (x.split(",")[0], x.split(",")[3]))
user_mapped.take(5)

[('1', '3'), ('2', '7'), ('3', '2'), ('4', '10'), ('5', '1')]

In [16]:
#Maper rating lay userId, rating
rating_mapped = ratings_rdd.map(lambda x: (x.split(",")[0], (float(x.split(",")[2]))))
rating_mapped.take(5)

[('7', 4.5), ('23', 3.5), ('45', 4.0), ('12', 3.0), ('38', 4.5)]

In [35]:
#join rating va user 
rating_user_joined = rating_mapped.join(user_mapped)
#map lai lay occupation id lam key
rating_user_joined = rating_user_joined.map(lambda x: (x[1][1], (float(x[1][0]), 1)))
rating_user_joined.take(5)


[('8', (3.0, 1)),
 ('8', (3.5, 1)),
 ('8', (4.0, 1)),
 ('8', (4.5, 1)),
 ('11', (4.5, 1))]

In [36]:
#join voi occupation_rdd de lay ten occupation
rating_occupation_joined = rating_user_joined.join(occupation_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1])))
#lay them occupation name lam key 
rating_occupation_joined = rating_occupation_joined.map(lambda x: (x[1][1], x[1][0]))
rating_occupation_joined.take(5)

[('Teacher', (4.5, 1)),
 ('Teacher', (4.0, 1)),
 ('Teacher', (3.5, 1)),
 ('Teacher', (3.0, 1)),
 ('Teacher', (3.5, 1))]

In [37]:
#tinh toan rating trung binh theo occupation
totalRating = rating_occupation_joined.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
averageRating = totalRating.mapValues(lambda x: (x[0] / x[1], x[1]))
averageRating.collect()

[('Doctor', (3.6904761904761907, 21)),
 ('Salesperson', (3.6470588235294117, 17)),
 ('Programmer', (4.25, 10)),
 ('Lawyer', (3.6470588235294117, 17)),
 ('Engineer', (3.5555555555555554, 18)),
 ('Artist', (3.727272727272727, 11)),
 ('Accountant', (3.5833333333333335, 6)),
 ('Designer', (4.0, 13)),
 ('Journalist', (3.8529411764705883, 17)),
 ('Student', (4.0, 8)),
 ('Consultant', (3.857142857142857, 14)),
 ('Nurse', (3.8636363636363638, 11)),
 ('Manager', (3.46875, 16)),
 ('Teacher', (3.7, 5))]

In [40]:
def format_result(record):
    occupation = record[0]
    data = record[1]
    avg_rating = data[0]
    total_ratings = data[1]
    return f"{occupation} - AverageRating : {avg_rating:.2f} (Total Ratings : {total_ratings})"

formatted_results = averageRating.map(format_result)
formatted_results.collect()

['Doctor - AverageRating : 3.69 (Total Ratings : 21)',
 'Salesperson - AverageRating : 3.65 (Total Ratings : 17)',
 'Programmer - AverageRating : 4.25 (Total Ratings : 10)',
 'Lawyer - AverageRating : 3.65 (Total Ratings : 17)',
 'Engineer - AverageRating : 3.56 (Total Ratings : 18)',
 'Artist - AverageRating : 3.73 (Total Ratings : 11)',
 'Accountant - AverageRating : 3.58 (Total Ratings : 6)',
 'Designer - AverageRating : 4.00 (Total Ratings : 13)',
 'Journalist - AverageRating : 3.85 (Total Ratings : 17)',
 'Student - AverageRating : 4.00 (Total Ratings : 8)',
 'Consultant - AverageRating : 3.86 (Total Ratings : 14)',
 'Nurse - AverageRating : 3.86 (Total Ratings : 11)',
 'Manager - AverageRating : 3.47 (Total Ratings : 16)',
 'Teacher - AverageRating : 3.70 (Total Ratings : 5)']